In [10]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/amansingh45/crab-age/crab/train.csv
/kaggle/input/datasets/amansingh45/crab-age/crab/test.csv
/kaggle/input/competitions/predict-crab-age-c-1-fall-2026-27/sample_submission.csv
/kaggle/input/competitions/predict-crab-age-c-1-fall-2026-27/train.csv
/kaggle/input/competitions/predict-crab-age-c-1-fall-2026-27/test.csv


In [11]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [12]:
class CrabDataset(Dataset):
    def __init__(self, features, targets=None):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.targets = torch.tensor(targets, dtype=torch.float32).unsqueeze(1) if targets is not None else None

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        if self.targets is not None:
            return self.features[idx], self.targets[idx]
        return self.features[idx]

In [13]:
class CrabAgeFFNN(nn.Module):
    def __init__(self, input_dim):
        super(CrabAgeFFNN, self).__init__()
        self.linear1 = nn.Linear(input_dim, 128)
        self.relu1 = nn.ReLU()
        
        self.linear2 = nn.Linear(128, 64)
        self.relu2 = nn.ReLU()
        
        self.linear3 = nn.Linear(64, 32)
        self.relu3 = nn.ReLU()
        
        self.output_layer = nn.Linear(32, 1) 

    def forward(self, x):
        x = self.linear1(x)
        x = self.relu1(x)
        x = self.linear2(x)
        x = self.relu2(x)
        x = self.linear3(x)
        x = self.relu3(x)
        x = self.output_layer(x)
        return x

In [14]:
def main():
    train_df = pd.read_csv("/kaggle/input/datasets/amansingh45/crab-age/crab/train.csv")
    test_df = pd.read_csv("/kaggle/input/datasets/amansingh45/crab-age/crab/test.csv")
    
    test_ids = test_df['id']
    
    y = train_df['Age'].values
    X = train_df.drop(['id', 'Age'], axis=1)
    X_test = test_df.drop(['id'], axis=1)
    
    X = pd.get_dummies(X, columns=['Sex'], drop_first=True)
    X_test = pd.get_dummies(X_test, columns=['Sex'], drop_first=True)
    
    X, X_test = X.align(X_test, join='left', axis=1, fill_value=0)
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_test_scaled = scaler.transform(X_test)
    
    X_train, X_val, y_train, y_val = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
    
    train_dataset = CrabDataset(X_train, y_train)
    val_dataset = CrabDataset(X_val, y_val)
    test_dataset = CrabDataset(X_test_scaled)
    
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
    
    input_dim = X_train.shape[1]
    model = CrabAgeFFNN(input_dim)
    
    criterion = nn.L1Loss() 
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    epochs = 100
    best_val_loss = float('inf')
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            predictions = model(batch_X)
            loss = criterion(predictions, batch_y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * batch_X.size(0)
            
        train_loss /= len(train_loader.dataset)
        
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                predictions = model(batch_X)
                loss = criterion(predictions, batch_y)
                val_loss += loss.item() * batch_X.size(0)
        val_loss /= len(val_loader.dataset)
        
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs} | Train MAE: {train_loss:.4f} | Val MAE: {val_loss:.4f}")
            
    model.eval()
    test_predictions = []
    with torch.no_grad():
        for batch_X in test_loader:
            preds = model(batch_X)
            test_predictions.extend(preds.numpy().flatten())
            
    submission = pd.DataFrame({
        'id': test_ids,
        'Age': test_predictions  
    })
    
    submission.to_csv("submission.csv", index=False)
    print("Predictions saved to submission.csv")

if __name__ == "__main__":
    main()

Epoch 10/100 | Train MAE: 1.3669 | Val MAE: 1.3494
Epoch 20/100 | Train MAE: 1.3475 | Val MAE: 1.3566
Epoch 30/100 | Train MAE: 1.3435 | Val MAE: 1.3110
Epoch 40/100 | Train MAE: 1.3306 | Val MAE: 1.3161
Epoch 50/100 | Train MAE: 1.3301 | Val MAE: 1.3226
Epoch 60/100 | Train MAE: 1.3270 | Val MAE: 1.3567
Epoch 70/100 | Train MAE: 1.3231 | Val MAE: 1.3236
Epoch 80/100 | Train MAE: 1.3232 | Val MAE: 1.3217
Epoch 90/100 | Train MAE: 1.3169 | Val MAE: 1.3108
Epoch 100/100 | Train MAE: 1.3146 | Val MAE: 1.3184
Predictions saved to submission.csv
